# Bank Telemarketing Campaign Effectiveness Analyzer

**Author:** Jasveer Singh

**Project:** IBM SkillsBuild / AI & Data Science Internship

This notebook analyzes the UCI Bank Marketing dataset using descriptive, diagnostic, predictive and prescriptive/business analytics. It is a structured adaptation of the source project by Sanket Kale; the source is acknowledged for transparency.

## 1. Imports and Configuration

In [ ]:
import io
import zipfile
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, classification_report, confusion_matrix, roc_curve

RANDOM_STATE = 42


## 2. Load the UCI Bank Marketing Dataset

The notebook first checks for a local `data/bank-additional-full.csv`. If it is not available, it downloads the official UCI archive.

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
LOCAL_PATH = 'data/bank-additional-full.csv'

def load_data():
    try:
        return pd.read_csv(LOCAL_PATH, sep=';')
    except FileNotFoundError:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            outer_bytes = response.read()
        with zipfile.ZipFile(io.BytesIO(outer_bytes)) as outer:
            inner_name = next(n for n in outer.namelist() if n.endswith('bank-additional.zip'))
            inner_bytes = outer.read(inner_name)
        with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner:
            csv_name = next(n for n in inner.namelist() if n.endswith('bank-additional-full.csv'))
            with inner.open(csv_name) as f:
                return pd.read_csv(f, sep=';')

df_raw = load_data()
print('Raw shape:', df_raw.shape)
display(df_raw.head())


## 3. Data Cleaning and Feature Engineering

In [ ]:
df = df_raw.copy()
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print('Duplicates removed:', before - len(df))

df = df.rename(columns={'emp.var.rate':'emp_var_rate','cons.price.idx':'cons_price_idx','cons.conf.idx':'cons_conf_idx','nr.employed':'nr_employed'})
df['y_binary'] = (df['y'] == 'yes').astype(int)

unknown_cols = ['job','marital','education','default','housing','loan','poutcome']
for col in unknown_cols:
    df[col] = df[col].replace('unknown', np.nan)
    mode = df[col].mode(dropna=True)
    if not mode.empty:
        df[col] = df[col].fillna(mode.iloc[0])

df['pdays'] = df['pdays'].replace(999, np.nan)
df['was_contacted_before'] = df['pdays'].notna().astype(int)
education_order = {'illiterate':0,'basic.4y':1,'basic.6y':2,'basic.9y':3,'high.school':4,'professional.course':5,'university.degree':6}
df['education_ord'] = df['education'].map(education_order).fillna(0).astype(int)
df['call_duration_min'] = (df['duration'] / 60).round(2)
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
dow_order = ['mon','tue','wed','thu','fri']
df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)
df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=dow_order, ordered=True)
print('Clean shape:', df.shape)


## 4. Descriptive Analytics

In [ ]:
total = len(df)
subscribed = int(df['y_binary'].sum())
subscription_rate = subscribed / total * 100
summary = pd.DataFrame({'Metric':['Total records','Subscribed','Not subscribed','Subscription rate (%)','Average age','Average call duration (min)'],'Value':[total,subscribed,total-subscribed,round(subscription_rate,2),round(df['age'].mean(),2),round(df['call_duration_min'].mean(),2)]})
display(summary)

fig = px.histogram(df, x='y', color='y', title='Subscription Outcome Distribution')
fig.show()

monthly = df.groupby('month', observed=True)['y_binary'].agg(['count','mean']).reset_index()
monthly['subscription_rate_pct'] = monthly['mean'] * 100
fig = px.bar(monthly, x='month', y='subscription_rate_pct', title='Subscription Rate by Month')
fig.show()


In [ ]:
job_rate = df.groupby('job', observed=True)['y_binary'].mean().mul(100).sort_values(ascending=False).reset_index(name='subscription_rate_pct')
display(job_rate)
fig = px.bar(job_rate, x='subscription_rate_pct', y='job', orientation='h', title='Subscription Rate by Job')
fig.show()

contact_rate = df.groupby('contact', observed=True)['y_binary'].mean().mul(100).reset_index(name='subscription_rate_pct')
display(contact_rate)


## 5. Diagnostic Analytics

Call duration is used for post-campaign diagnosis but excluded from the pre-call predictive model to avoid target leakage.

In [ ]:
duration_summary = df.groupby('y')['call_duration_min'].agg(['mean','median','count']).round(2)
display(duration_summary)

poutcome_rate = df.groupby('poutcome', observed=True)['y_binary'].mean().mul(100).sort_values(ascending=False).reset_index(name='subscription_rate_pct')
display(poutcome_rate)

campaign_rate = df.groupby('campaign')['y_binary'].mean().mul(100).reset_index(name='subscription_rate_pct')
campaign_rate = campaign_rate[campaign_rate['campaign'] <= 10]
fig = px.line(campaign_rate, x='campaign', y='subscription_rate_pct', markers=True, title='Subscription Rate by Number of Contacts')
fig.show()


## 6. Predictive Analytics — Random Forest

In [ ]:
numeric_features = ['age','education_ord','was_contacted_before','campaign','previous','emp_var_rate','cons_price_idx','cons_conf_idx','euribor3m','nr_employed']
categorical_features = ['job','marital','housing','loan','contact','month','day_of_week','poutcome']
model_df = df[numeric_features + categorical_features].copy()
model_df['month'] = model_df['month'].astype(str)
model_df['day_of_week'] = model_df['day_of_week'].astype(str)
X = pd.get_dummies(model_df, columns=categorical_features, drop_first=True).astype(int)
y = df['y_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
model = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:,1]
pred = (proba >= 0.5).astype(int)
metrics = {'Accuracy':accuracy_score(y_test,pred),'F1':f1_score(y_test,pred),'ROC-AUC':roc_auc_score(y_test,proba),'Average Precision':average_precision_score(y_test,proba)}
display(pd.DataFrame([metrics]).round(4))
print(classification_report(y_test, pred, digits=4))


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC-AUC = {metrics["ROC-AUC"]:.4f}'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random baseline'))
fig.update_layout(title='ROC Curve', xaxis_title='False Positive Rate', yaxis_title='True Positive Rate')
fig.show()

cm = confusion_matrix(y_test, pred)
fig = px.imshow(cm, text_auto=True, title='Confusion Matrix', labels={'x':'Predicted','y':'Actual'})
fig.show()

importance = pd.DataFrame({'feature':X.columns,'importance':model.feature_importances_}).sort_values('importance', ascending=False).head(20)
fig = px.bar(importance.sort_values('importance'), x='importance', y='feature', orientation='h', title='Top 20 Feature Importances')
fig.show()


## 7. Business Insights and Recommendations

- The target class is imbalanced, so ROC-AUC, F1 and Average Precision should be considered alongside accuracy.
- Previous campaign outcome and contact characteristics can help identify different response segments.
- Call duration is strongly associated with the historical outcome but is not suitable for pre-call scoring because it is known only after contact.
- Contact frequency can be analyzed for diminishing response rates.
- Economic indicators provide historical context for campaign performance.

Recommendations should be validated against current campaign costs, policies and new data before operational use.

## 8. Limitations

1. The dataset represents historical campaigns from a specific Portuguese banking institution and period.
2. Observational relationships do not by themselves establish causation.
3. `duration` is excluded from the pre-call model because it is unavailable before a call.
4. Model performance may change on new populations or economic conditions.
5. Threshold selection should reflect the costs of false positives and false negatives.

## 9. References and Attribution

- UCI Machine Learning Repository — Bank Marketing Dataset.
- Moro, S., Cortez, P., & Rita, P. (2014). *A Data-Driven Approach to Predict the Success of Bank Telemarketing*. Decision Support Systems.


In [ ]:
print('Notebook prepared for: Jasveer Singh')
print('Project: Bank Telemarketing Campaign Effectiveness Analyzer')
